<a href="https://colab.research.google.com/github/matzz-11/Quantum_Colab.ipynb/blob/main/8_%C3%81tomo_de_Hidrog%C3%AAnio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Código para visualização das superfícies e cortes internos das densidades de probabilidade para função de onda do átomo de Hidrogênio:

In [ ]:
# Bibliotecas utilizadas
import numpy as np # Cálculos
import matplotlib.pyplot as plt # Criação de gráficos
from scipy.special import sph_harm_y, genlaguerre # Funções especiais
from math import factorial # Operações
from skimage.measure import marching_cubes # Geometria da superfície 3D
import ipywidgets as widgets # Widgets
from IPython.display import display # Renderizar interfaces

# Função da Parte Radial
def R_nl(n, l, r):
    rho = 2.0 * r / n
    norm = np.sqrt((2.0 / n)**3 * factorial(n - l - 1) / (2.0 * n * factorial(n + l)))
    L = genlaguerre(n - l - 1, 2 * l + 1)
    return norm * np.exp(-rho / 2.0) * (rho ** l) * L(rho)

# Função de Plotagem Principal
def plot_orbital_completo(n, l, m):
    # Valor fixo padrão para a densidade da isossuperfície (%)
    limiar_percentual = 2.0

    # O tamanho do orbital cresce proporcional a n^2
    limite = n**2 * 4.5

    fig = plt.figure(figsize=(15, 6))

    # Corte 2D
    ax1 = fig.add_subplot(121)

    res_2d = 400
    x_2d = np.linspace(-limite, limite, res_2d)
    z_2d = np.linspace(-limite, limite, res_2d)
    X2, Z2 = np.meshgrid(x_2d, z_2d)

    R2 = np.sqrt(X2**2 + Z2**2)
    R2 = np.maximum(R2, 1e-10)
    Theta2 = np.arccos(Z2 / R2)

    Phi2 = np.zeros_like(X2)
    Phi2[X2 < 0] = np.pi

    Psi_2D = R_nl(n, l, R2) * sph_harm_y(l, m, Theta2, Phi2)
    Prob_2D = np.abs(Psi_2D)**2

    # Usando o colormap 'turbo' para cores extremamente vivas
    im = ax1.imshow(Prob_2D, extent=[-limite, limite, -limite, limite],
                    origin='lower', cmap='turbo', interpolation='bicubic')

    ax1.set_title(f'Corte Interno $|\\psi|^2$\n({n}, {l}, {m})', fontsize=14, pad=15)
    ax1.set_xlabel('X ($a_0$)')
    ax1.set_ylabel('Z ($a_0$)')
    fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, label='Densidade Probabilidade')

    # Isossuperfície 3D
    ax2 = fig.add_subplot(122, projection='3d')

    res_3d = 60
    x_3d = np.linspace(-limite, limite, res_3d)
    y_3d = np.linspace(-limite, limite, res_3d)
    z_3d = np.linspace(-limite, limite, res_3d)
    X3, Y3, Z3 = np.meshgrid(x_3d, y_3d, z_3d, indexing='ij')

    R3 = np.sqrt(X3**2 + Y3**2 + Z3**2)
    R3 = np.maximum(R3, 1e-10)
    Theta3 = np.arccos(Z3 / R3)
    Phi3 = np.arctan2(Y3, X3)

    Psi_3D = R_nl(n, l, R3) * sph_harm_y(l, m, Theta3, Phi3)
    Prob_3D = np.abs(Psi_3D)**2

    isovalue = np.max(Prob_3D) * (limiar_percentual / 100.0)

    try:
        verts, faces, _, _ = marching_cubes(Prob_3D, level=isovalue,
                                            spacing=(x_3d[1]-x_3d[0], y_3d[1]-y_3d[0], z_3d[1]-z_3d[0]))
        verts = verts - limite

        # Superfície com colormap vibrante e grade fina
        ax2.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                         cmap='turbo', edgecolor='black', linewidth=0.2, alpha=0.9, antialiased=True)
    except Exception:
        ax2.text2D(0.5, 0.5, f"Isossuperfície vazia\n(Limiar fixo em {limiar_percentual}%).",
                   ha='center', va='center', transform=ax2.transAxes)

    ax2.set_title(f'Superfície Externa $|\\psi|^2$\n({n}, {l}, {m})', fontsize=14, pad=15)

    ax2.set_xlim([-limite, limite])
    ax2.set_ylim([-limite, limite])
    ax2.set_zlim([-limite, limite])

    ax2.xaxis.pane.fill = False
    ax2.yaxis.pane.fill = False
    ax2.zaxis.pane.fill = False
    ax2.xaxis.pane.set_edgecolor('w')
    ax2.yaxis.pane.set_edgecolor('w')
    ax2.zaxis.pane.set_edgecolor('w')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

# Construção dos Widgets
n_slider = widgets.IntSlider(min=1, max=10, step=1, value=4, description='n (Principal):')
l_slider = widgets.IntSlider(min=0, max=9, step=1, value=2, description='l (Azimutal):')
m_slider = widgets.IntSlider(min=-9, max=9, step=1, value=0, description='m (Magnético):')

def atualizar_limites_l(*args):
    l_slider.max = n_slider.value - 1
    if l_slider.value > n_slider.value - 1:
        l_slider.value = n_slider.value - 1

def atualizar_limites_m(*args):
    m_slider.min = -l_slider.value
    m_slider.max = l_slider.value
    if m_slider.value < -l_slider.value:
        m_slider.value = -l_slider.value
    if m_slider.value > l_slider.value:
        m_slider.value = l_slider.value

n_slider.observe(atualizar_limites_l, 'value')
l_slider.observe(atualizar_limites_m, 'value')

atualizar_limites_l()
atualizar_limites_m()

# Interface agrupando os 3 números quânticos
ui = widgets.VBox([n_slider, l_slider, m_slider])
out = widgets.interactive_output(plot_orbital_completo,
                                 {'n': n_slider, 'l': l_slider, 'm': m_slider})

display(ui, out)